First we import stuff

In [ ]:
from pandapower.control import ConstControl

import pandapipes as pp

import numpy as np
import copy
import matplotlib.pyplot as plt

import tempfile
# create empty net
import pandas as pd
from pandapipes.timeseries import run_timeseries, init_default_outputwriter
from pandapower.timeseries import OutputWriter, DFData

Then we define our Outputwriter, which logs and feedsback data during a timeseries. In a future release this will come as a default version.

In [ ]:
class OutputWriterTransient(OutputWriter):
    def _save_single_xls_sheet(self, append):
        raise NotImplementedError("Sorry not implemented yet")

    def _init_log_variable(self, net, table, variable, index=None, eval_function=None,
                           eval_name=None):
        if table == "res_internal":
            index = np.arange(len(net.junction) + (net.pipe.sections-1).sum())
        return super()._init_log_variable(net, table, variable, index, eval_function, eval_name)


def _output_writer(net, time_steps, ow_path=None):
    """
    Creating an output writer.

    :param net: Prepared pandapipes net
    :type net: pandapipesNet
    :param time_steps: Time steps to calculate as a list or range
    :type time_steps: list, range
    :param ow_path: Path to a folder where the output is written to.
    :type ow_path: string, default None
    :return: Output writer
    :rtype: pandapower.timeseries.output_writer.OutputWriter
    """

    if transient_transfer:
        log_variables = [
            ('res_junction', 't_k'), ('res_junction', 'p_bar'), ('res_pipe', 't_to_k'), ('res_internal', 't_k')
        ]
    else:
        log_variables = [
            ('res_junction', 't_k'), ('res_junction', 'p_bar'), ('res_pipe', 't_to_k')
        ]
    ow = OutputWriterTransient(net, time_steps, output_path=ow_path, log_variables=log_variables)
    return ow

We create a simple network consisting of a pipe that connects an external grid with a sink

In [ ]:
net = pp.create_empty_network(fluid="water")
# create junctions
j1 = pp.create_junction(net, pn_bar=1.05, tfluid_k=293, name="Junction 1")
j2 = pp.create_junction(net, pn_bar=1.05, tfluid_k=293, name="Junction 2")

# create junction elements
ext_grid = pp.create_ext_grid(net, junction=j1, p_bar=5, t_k=330, name="Grid Connection")
sink = pp.create_sink(net, junction=j2, mdot_kg_per_s=2, name="Sink")
print(net)

For the pipes we define sections, since those are necessary for transient calculations to simulate the pipe as a series of finite elements.

In [ ]:
# create branch elements
sections = 74
nodes = 2
length = 1
pp.create_pipe_from_parameters(net, j1, j2, length, 75e-3, k_mm=.0472, sections=sections,
                               u_w_per_m2k=50, text_k=293)
print(net.pipe)

Next we define the input data for the timeseries calculation. In this case the temperature at the external grid.

In [ ]:
ds = DFData(pd.DataFrame({"t_k": [330] * 50 + [350] * 100 + [330] * 500}))
t_ctrl = ConstControl(net, "ext_grid", "t_k", 0, profile_name="t_k", data_source=ds)

And the number of timesteps and their length we want to simulate.

In [ ]:
time_steps = range(300)
dt = 5

Then we define the outputwriter and run the simulation

In [ ]:
transient_transfer=True
ow = _output_writer(net, time_steps, ow_path=tempfile.gettempdir())
run_timeseries(net, time_steps, dynamic_sim=True, transient=transient_transfer, mode="sequential", dt=dt,
               reuse_internal_data=True)

Now we can extract the data

In [ ]:
res_T = ow.np_results["res_internal.t_k"]
res_T_df = pd.DataFrame(res_T)
#It can for example be saved to excel
res_T_df.to_excel('res_T.xlsx')

We prepare plotting

In [ ]:
pipe1 = np.zeros(((sections + 1), res_T.shape[0]))
pipe1[0, :] = copy.deepcopy(res_T[:, 0])
pipe1[-1, :] = copy.deepcopy(res_T[:, 1])
pipe1[1:-1, :] = np.transpose(copy.deepcopy(res_T[:, nodes:nodes + (sections - 1)]))
print(pipe1)

In [ ]:
import matplotlib.animation as animation

fig, ax = plt.subplots()
x = np.arange(0, sections + 1, 1) * length * 1000 / sections
line, = ax.plot([], [], lw=2)
ax.set_xlim(0, x[-1])
ax.set_ylim(280, 355)
ax.set_title("Temperature Profile Over Time")
ax.set_xlabel("Length [mm]")
ax.set_ylabel("Temperature [K]")

def init():
    line.set_data([], [])
    return line,

def update(frame):
    line.set_data(x, pipe1[:, frame])
    ax.set_title(f"Temperature Profile - Time Step {frame}")
    return line,


ani = animation.FuncAnimation(
    fig,
    update,
    frames=range(0, pipe1.shape[1], 5),  # every 5th timestep
    init_func=init,
    blit=True,
    repeat=False,
    interval=500  # <-- this sets the delay to 500ms (0.5 seconds) between frames
)


plt.show(block=True)